# 原始预测到 f696 提案的 EMA/HOLD 身份桥

只读三个封存 Qlib Recorder，对照原始预测、来源目标及 f696 五输出；收益仅按冻结来源算法更新下一日虚拟持仓。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from src.workflow.pit_constituent_auxiliary_training_allocation import PitAuxiliaryAllocationSpec,AuxiliaryTrainingCandidate,_smooth_weights

root=Path.cwd();card=root/"research/stage1_f696_raw_to_target_bridge_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());src=plan["sources"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and all(v==0 for v in plan["budget"].values())
a=root/src["action_card"];p=root/src["parent_card"];f=root/src["f696_card"]
assert sha(a/"summary.json")==src["action_summary_sha256"] and sha(a/"nas_readback.json")==src["action_readback_sha256"]
assert sha(p/"summary.json")==src["parent_summary_sha256"] and sha(f/"summary.json")==src["f696_summary_sha256"]
sa=json.loads((a/"summary.json").read_text());ra=json.loads((a/"nas_readback.json").read_text())
sp=json.loads((p/"summary.json").read_text());sf=json.loads((f/"summary.json").read_text())
assert sa["nas_evidence"]["sha256"]==src["action_archive_sha256"] and sa["nas_evidence"]["size_bytes"]==src["action_archive_size_bytes"]
assert ra["NAS_archive_sha256"]==src["action_archive_sha256"] and ra["artifact_hashes_checked"]>=28
assert sp["nas_evidence"]["sha256"]==src["parent_archive_sha256"] and sp["nas_evidence"]["size_bytes"]==src["parent_archive_size_bytes"]
assert sf["nas_evidence"]["sha256"]==src["f696_archive_sha256"] and sf["nas_evidence"]["size_bytes"]==src["f696_archive_size_bytes"]
for name,digest in src["code_sha256"].items():assert sha(root/name)==digest
ar=a/"qlib_recorder"/src["action_experiment_id"]/src["action_recorder_id"]/"artifacts"
pr=p/"qlib_recorder"/src["parent_experiment_id"]/src["parent_recorder_id"]/"artifacts"
fr=f/"qlib_recorder"/src["f696_experiment_id"]/src["f696_recorder_id"]/"artifacts"
for name,digest in src["action_artifacts"].items():assert sha(ar/name)==digest,name
for name,digest in src["parent_artifacts"].items():assert sha(pr/name)==digest,name
assert sha(fr/"inputs/frame")==src["f696_frame_sha256"]
assert sha(root/src["provider"]/"qlib_store_manifest.json")==src["provider_manifest_sha256"]
qlib.init(provider_uri=str((root/src["provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None)
with R.uri_context(str(a/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["action_experiment_id"],recorder_id=src["action_recorder_id"])
 action={name:rec.load_object(name) for name in src["action_artifacts"]}
with R.uri_context(str(p/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["parent_experiment_id"],recorder_id=src["parent_recorder_id"])
 parent={name:rec.load_object(name) for name in src["parent_artifacts"]}
with R.uri_context(str(f/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["f696_experiment_id"],recorder_id=src["f696_recorder_id"])
 frame=rec.load_object("inputs/frame")
market=action["inputs/market"]
dates=pd.DatetimeIndex(market.signal_date)
assert len(dates)==944 and dates.is_unique and dates.equals(pd.DatetimeIndex(frame.signal_date))
raw=action["inputs/raw_predictions"]["frozen_current"].reindex(dates)
target=action["inputs/source_targets"]["frozen_current"].reindex(dates)
columns=["soxl_weight","soxx_weight","cash_weight","tail_probability","alpha_probability"]
assert list(raw.columns)==columns and list(target.columns)==columns
assert raw[columns].notna().all().all() and target[columns].notna().all().all()
assert len(pd.DatetimeIndex(D.calendar(start_time="2011-01-03",end_time="2026-10-09",freq="day")))==3966
parent_plan=parent["inputs/plan"]
spec=PitAuxiliaryAllocationSpec(root=root,smoothing_span=parent_plan["smoothing_span"],rebalance_threshold=parent_plan["rebalance_threshold"])
candidate=AuxiliaryTrainingCandidate(**parent["inputs/frozen_parent_models"]["outer_refit"]["candidate"])
returns=market.set_index("signal_date")[["soxl_return","soxx_return"]]
assert returns.index.equals(dates) and np.isfinite(returns.to_numpy(float)).all()
replayed=_smooth_weights(raw,spec=spec,candidate=candidate,asset_returns=returns)
frame_outputs=frame.set_index(pd.DatetimeIndex(frame.signal_date)).loc[:,columns]
max_replay_target={c:float((replayed[c]-target[c]).abs().max()) for c in columns}
max_target_f696={c:float((target[c]-frame_outputs[c]).abs().max()) for c in columns}
pass_gate=all(v<=1e-7 for v in [*max_replay_target.values(),*max_target_f696.values()])
weights=columns[:3]
span=candidate.smoothing_span if candidate.smoothing_span is not None else spec.smoothing_span
ema=raw[weights].ewm(span=span,adjust=False).mean()
raw_ema={c:{"mean_abs":float((raw[c]-ema[c]).abs().mean()),"max_abs":float((raw[c]-ema[c]).abs().max())} for c in weights}
ema_accepted={c:{"mean_abs":float((ema[c]-replayed[c]).abs().mean()),"max_abs":float((ema[c]-replayed[c]).abs().max())} for c in weights}
raw_accepted={c:{"mean_abs":float((raw[c]-replayed[c]).abs().mean()),"max_abs":float((raw[c]-replayed[c]).abs().max())} for c in weights}
non_ema=(replayed[weights]-ema[weights]).abs().max(axis=1)>1e-10
perturbed=returns.copy();perturbed.iloc[0,0]=float(perturbed.iloc[0,0])+0.10
perturbed_target=_smooth_weights(raw,spec=spec,candidate=candidate,asset_returns=perturbed)
first_day_perturbation_max=float((perturbed_target.iloc[0][columns]-replayed.iloc[0][columns]).abs().max())
assert first_day_perturbation_max<1e-12
annual=[]
for year in (2023,2024,2025,2026):
 mask=dates.year==year
 annual.append({"year":year,"sessions":int(mask.sum()),"partial_endpoint":year in (2023,2026),
                "mean_raw_cash":float(raw.loc[mask,"cash_weight"].mean()),"mean_accepted_cash":float(replayed.loc[mask,"cash_weight"].mean()),
                "mean_accepted_minus_raw_cash":float((replayed.loc[mask,"cash_weight"]-raw.loc[mask,"cash_weight"]).mean()),
                "non_ema_accepted_days":int(non_ema[mask].sum())})
result={"status":"exact_raw_to_f696_target_bridge_pass" if pass_gate else "raw_to_f696_target_bridge_failed",
        "plan_sha256":sha(plan_path),"sessions":len(dates),"f696_rows":len(frame),
        "smoothing_span":span,"rebalance_threshold":candidate.rebalance_threshold if candidate.rebalance_threshold is not None else spec.rebalance_threshold,
        "replay_to_source_target_max_abs":max_replay_target,"source_target_to_f696_max_abs":max_target_f696,"identity_gate_pass":pass_gate,
        "raw_to_ema":raw_ema,"ema_to_accepted":ema_accepted,"raw_to_accepted":raw_accepted,
        "accepted_differs_from_ema_days_lower_bound_hold":int(non_ema.sum()),"annual":annual,
        "first_day_return_perturbation_target_day_max_abs":first_day_perturbation_max,
        "new_market_requests":0,"new_model_fits":0,"new_native_accounts":0,"new_bootstraps":0,"new_recorders":0,
        "economic_factor_qualified":False,"independent_validation":False,
        "limits":["EMA/HOLD reuses the historical source market return path to update virtual gross holdings only for later targets",
                  "Non-EMA accepted count is a lower bound on HOLD, not a re-executed funded strategy",
                  "Source-target identity does not qualify model input sensitivity or SOXX economic success"]}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"status":result["status"],"max_replay_target":max_replay_target,"max_target_f696":max_target_f696,"raw_to_accepted_cash":raw_accepted["cash_weight"],"non_ema_days":int(non_ema.sum())})


2026/10/24 16:23:36 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[57673:MainThread](2026-10-24 16:23:37,938) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[57673:MainThread](2026-10-24 16:23:37,940) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[57673:MainThread](2026-10-24 16:23:37,940) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{'status': 'exact_raw_to_f696_target_bridge_pass', 'max_replay_target': {'soxl_weight': 0.0, 'soxx_weight': 0.0, 'cash_weight': 0.0, 'tail_probability': 0.0, 'alpha_probability': 0.0}, 'max_target_f696': {'soxl_weight': 0.0, 'soxx_weight': 0.0, 'cash_weight': 0.0, 'tail_probability': 0.0, 'alpha_probability': 0.0}, 'raw_to_accepted_cash': {'mean_abs': 0.035806897395000516, 'max_abs': 0.2359998534321116}, 'non_ema_days': 322}


## 中文结论

只读封存的原始模型预测、来源目标、Qlib 来源市场收益和原 f696 提案。按保存的 3 日 EMA、2% 再平衡门槛与虚拟毛值持仓路径重放，944 个信号日的 SOXL、SOXX、现金、alpha、tail 五输出对 `source_targets` 与 f696 `inputs/frame` 的最大绝对差全部为 **0**。此前裸模型前向只吻合 alpha/tail、未吻合权重的依赖原因已定位；原始模型并非账户直接接收的配置目标。

现金权重原始预测到最终提案的平均／最大绝对变化是 **0.03581 / 0.23600**；原始到 EMA 为 **0.03301 / 0.23600**，EMA 到接受目标为 **0.00321 / 0.01987**。接受目标有322日与 EMA 不同，这是 HOLD 的下界，不把它冒充账户交易次数。各年最终减原始的平均现金权重依次为2023 **+0.00073**、2024 **-0.00278**、2025 **+0.00333**、2026 **-0.00345**，未见单方向的年度现金推升。固定扰动首日 SOXL 收益不改变首日目标，支持来源算法在当日接受目标后才用当日收益更新下一日虚拟持仓；这不认证历史实时送达。

决定 `exact_raw_to_f696_target_bridge_pass_source_lineage_only`。以后若分析原39项输入对**实际 f696 目标**的敏感度，必须通过完整模型前向和此 EMA/HOLD 链，不能拿裸模型权重代替。此处只有来源身份，0拟合、0新账户、0独立验证，未证明某个因子能改善 SOXX 净超额。下一项输入敏感度需另行冻结，且其数值变化仍不是经济收益。
